In [3]:
import pandas as pd
import os

### Writing a CSV (to_csv)
- `df.to_csv("name.csv", index=False)` saves the table as a text file in the notebook's folder. It returns nothing.
- Always `index=False`; without it the row labels become an extra column that returns as `Unnamed: 0` when read back.
- Missing values are written as empty spaces and come back as NaN. `na_rep="missing"` writes a word instead.
- Save some columns: `df[["a","b"]].to_csv(...)`. Save what you did: `df.sort_values("x").to_csv(...)`.
- Other options: `sep=";"` (read it back with `sep=";"`), `header=False`.
- `pd.read_csv("name.csv")` reads it back. `os.getcwd()` shows the folder; `os.path.exists("name.csv")` checks that the file is there.
- Watch out: `df = df.to_csv(...)` makes `df` None; a wrong name or folder gives FileNotFoundError; a wrong `sep` gives one big column.

In [4]:
df = pd.DataFrame({"name":   ["Asha", "Ravi", "Meena"],
                   "city":   ["Indore", None, "Pune"],
                   "age":    [25, 31, None],
                   "salary": [50000, 62000, 45000]})

df.to_csv("people.csv", index=False)

In [6]:
os.getcwd()
os.path.exists("people.csv")

True

In [7]:
back = pd.read_csv("people.csv")
print(back)

    name    city   age  salary
0   Asha  Indore  25.0   50000
1   Ravi     NaN  31.0   62000
2  Meena    Pune   NaN   45000


In [8]:
df.to_csv("people_wrong.csv")
w = pd.read_csv("people_wrong.csv")
print(w.columns.tolist())      # ['Unnamed: 0', 'name', 'city', 'age', 'salary']

['Unnamed: 0', 'name', 'city', 'age', 'salary']


In [10]:


df = pd.DataFrame({"name":   ["Asha", "Ravi", "Meena"],
                   "city":   ["Indore", None, "Pune"],
                   "age":    [25, 31, None],
                   "salary": [50000, 62000, 45000]})

# step 1: save
df.to_csv("people.csv", index=False)
print(os.getcwd())                       # the folder where the file is saved
print(os.path.exists("people.csv"))      # True
print(open("people.csv").read())         # the text inside the file

# step 2: read it back
back = pd.read_csv("people.csv")
print(back)
print(back.dtypes)
print(back.equals(df))                   # True

# step 3: the index=False trap
df.to_csv("people_wrong.csv")            # no index=False
w = pd.read_csv("people_wrong.csv")
print(w.columns.tolist())                # ['Unnamed: 0', 'name', ...]
print(w.drop(columns="Unnamed: 0"))      # the fix

# step 4: other ways to save
df[["name", "salary"]].to_csv("small.csv", index=False)
print(pd.read_csv("small.csv"))
df.sort_values("salary").to_csv("sorted.csv", index=False)
df.to_csv("semi.csv", index=False, sep=";")
print(pd.read_csv("semi.csv").columns.tolist())             # one big column (wrong sep)
print(pd.read_csv("semi.csv", sep=";").columns.tolist())    # correct
df.to_csv("nohead.csv", index=False, header=False)
df.to_csv("na.csv", index=False, na_rep="missing")
print(open("na.csv").read())

# fail on purpose (remove the # to try one at a time)
# pd.read_csv("nofile.csv")             # FileNotFoundError: wrong name or folder
# df = df.to_csv("x.csv", index=False)  # df is now None

c:\Users\balar.CHETAN\Documents\DataEngineering\python-practice\data-analysis\pandas\notebooks
True
name,city,age,salary
Asha,Indore,25.0,50000
Ravi,,31.0,62000
Meena,Pune,,45000

    name    city   age  salary
0   Asha  Indore  25.0   50000
1   Ravi     NaN  31.0   62000
2  Meena    Pune   NaN   45000
name       object
city       object
age       float64
salary      int64
dtype: object
True
['Unnamed: 0', 'name', 'city', 'age', 'salary']
    name    city   age  salary
0   Asha  Indore  25.0   50000
1   Ravi     NaN  31.0   62000
2  Meena    Pune   NaN   45000
    name  salary
0   Asha   50000
1   Ravi   62000
2  Meena   45000
['name;city;age;salary']
['name', 'city', 'age', 'salary']
name,city,age,salary
Asha,Indore,25.0,50000
Ravi,missing,31.0,62000
Meena,Pune,missing,45000



### Block 2: read_csv options
- `usecols=["a","b"]`: read only these columns
- `nrows=n`: read only the first n rows (quick peek)
- `parse_dates=["date"]`: text becomes real dates
- `dtype={"id": str}`: force a type (ids, phone numbers, pin codes)
- `index_col="id"`: use a column as the row label
- `skiprows=n`: skip junk lines at the top
- `na_values=["-","NA_"]`: treat these words as missing (NaN)
- `header=None, names=[...]`: file has no header line, give your own names

Watch out:
- Column names in `usecols`/`parse_dates` must match exactly, else ValueError
- No header line but forgot `header=None`: the first data row becomes the column names
- A "-" in a number column makes it text; use `na_values` so it stays a number

In [11]:
orders = pd.DataFrame({"id":   [101, 102, 103, 104],
                       "city": ["Indore", "Pune", "Indore", "Delhi"],
                       "date": ["2026-10-01", "2026-10-02", "2026-10-03", "2026-10-04"],
                       "amount": [500, 700, 300, 900]})
orders.to_csv("orders.csv", index=False)

In [13]:
print(pd.read_csv("orders.csv", usecols=["city","amount"]))
print(pd.read_csv("orders.csv", nrows=2))

     city  amount
0  Indore     500
1    Pune     700
2  Indore     300
3   Delhi     900
    id    city        date  amount
0  101  Indore  2026-10-01     500
1  102    Pune  2026-10-02     700


In [15]:
print(pd.read_csv("orders.csv").dtypes)  
print(pd.read_csv("orders.csv", parse_dates=["date"]).dtypes)  

id         int64
city      object
date      object
amount     int64
dtype: object
id                 int64
city              object
date      datetime64[ns]
amount             int64
dtype: object


In [19]:
print(pd.read_csv("orders.csv", dtype={"id":str}).dtypes)

id        object
city      object
date      object
amount     int64
dtype: object


### Block 3: Check after reading, other formats
- After every read: `shape`, `head()`, `info()`, `isna().sum()`
- `info()` shows types and non-null counts, so you spot missing values and wrong types early
- JSON: `df.to_json("f.json", orient="records")`, `pd.read_json("f.json")`
- Excel: `df.to_excel("f.xlsx", index=False)`, `pd.read_excel("f.xlsx")` (needs openpyxl)
- Use the reader that matches the file type: csv -> read_csv, xlsx -> read_excel, json -> read_json

Watch out:
- `read_excel` needs `openpyxl`, else ImportError
- Always `index=False` in `to_excel` too
- Do not read an .xlsx file with `read_csv`

In [20]:
import pandas as pd

orders = pd.DataFrame({"id":   [101, 102, 103, 104],
                       "city": ["Indore", "Pune", None, "Delhi"],
                       "amount": [500, 700, 300, None]})
orders.to_csv("orders.csv", index=False)

# Part A: the check after reading
df = pd.read_csv("orders.csv")
print(df.shape)            # (rows, columns)
print(df.head(2))          # first 2 rows
df.info()                  # types + non-null counts
print(df.isna().sum())     # missing per column

# Part B (optional): JSON
orders.to_json("orders.json", orient="records")
print(open("orders.json").read())
print(pd.read_json("orders.json"))

# Part B (optional): Excel (needs openpyxl)
orders.to_excel("orders.xlsx", index=False)
print(pd.read_excel("orders.xlsx"))

(4, 3)
    id    city  amount
0  101  Indore   500.0
1  102    Pune   700.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      4 non-null      int64  
 1   city    3 non-null      object 
 2   amount  3 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 228.0+ bytes
id        0
city      1
amount    1
dtype: int64
[{"id":101,"city":"Indore","amount":500.0},{"id":102,"city":"Pune","amount":700.0},{"id":103,"city":null,"amount":300.0},{"id":104,"city":"Delhi","amount":null}]
    id    city  amount
0  101  Indore   500.0
1  102    Pune   700.0
2  103    None   300.0
3  104   Delhi     NaN
    id    city  amount
0  101  Indore   500.0
1  102    Pune   700.0
2  103     NaN   300.0
3  104   Delhi     NaN
